### Crear datos de prueba

In [0]:
%python
import os
import pandas as pd
from datetime import datetime

# Definir la ruta de nuestro volumen raw 
raw_path = "/Volumes/dev_casopractico/bronze/raw"
os.makedirs(raw_path, exist_ok=True)

# 1. Generar Datos de Clientes (con un ID corrupto o nulo)
clientes_data = {
    "cliente_id": ["C001", "C002", "C003", "ERROR_ROW", "C004"],
    "nombre": ["Juan Pérez", "Ana Gómez", "Luis Martínez", None, "María López"],
    "email": ["juan@email.com", "ana@email.com", "luis@email.com", "malformado", "maria@email.com"]
}
df_clientes = pd.DataFrame(clientes_data)
df_clientes.to_csv(f"{raw_path}/clientes_20261009.csv", index=False)

# 2. Generar Datos de Ventas - Día 1 (Con columna 'tienda' y una fila rota)
ventas_dia1 = {
    "venta_id": ["V001", "V002", "V003", "V004", "CORRUPT_VENTA"],
    "cliente_id": ["C001", "C002", "C001", "C003", None],
    "tienda_id": ["TIENDA_MADRID_01", "TIENDA_BARCELONA_01", "TIENDA_MADRID_01", "TIENDA_VALENCIA_01", "TIENDA_MADRID_01"],
    "monto": [150.50, 45.00, 99.99, "CIEN_EUROS", 20.00], # "CIEN_EUROS" romperá el tipo de dato numérico en Silver
    "fecha": ["2026-10-08", "2026-10-08", "2026-10-08", "2026-10-08", "2026-10-08"]
}
df_ventas1 = pd.DataFrame(ventas_dia1)
df_ventas1.to_csv(f"{raw_path}/ventas_20261008.csv", index=False)

print("Esquemas simulados y archivos CSV iniciales creados exitosamente en el volumen raw.")


In [0]:
%python
import os
import pandas as pd

raw_path = "/Volumes/dev_casopractico/bronze/raw"
os.makedirs(raw_path, exist_ok=True)

# ── CSV 1: Productos (con un precio nulo y una categoría vacía) ──
productos = {
    "producto_id": ["P001", "P002", "P003", "P004", "P005"],
    "nombre": ["Laptop", "Mouse", "Teclado", "Monitor", "Webcam"],
    "categoria": ["Electrónica", "Accesorios", "Accesorios", "", "Accesorios"],
    "precio": [1200.00, 25.50, 45.00, None, 60.00]
}
pd.DataFrame(productos).to_csv(f"{raw_path}/productos_20261009.csv", index=False)

# ── CSV 2: Ventas – Día 2 (nuevas filas, una con monto no numérico) ──
ventas_dia2 = {
    "venta_id": ["V005", "V006", "V007", "V008", "V009"],
    "cliente_id": ["C002", "C003", "C004", "C001", "C002"],
    "tienda_id": ["TIENDA_SEVILLA_01", "TIENDA_MADRID_01", "TIENDA_BARCELONA_01",
                  "TIENDA_VALENCIA_01", "TIENDA_SEVILLA_01"],
    "monto": [200.00, 15.75, 80.00, "CERO", 300.00],
    "fecha": ["2026-10-09", "2026-10-09", "2026-10-09", "2026-10-09", "2026-10-09"]
}
pd.DataFrame(ventas_dia2).to_csv(f"{raw_path}/ventas_20261009.csv", index=False)

# ── CSV 3: Ventas – Día 3 (un cliente_id inexistente y una fila duplicada) ──
ventas_dia3 = {
    "venta_id": ["V010", "V011", "V012", "V012", "V013"],
    "cliente_id": ["C001", "C999", "C003", "C003", "C004"],
    "tienda_id": ["TIENDA_MADRID_01", "TIENDA_BILBAO_01", "TIENDA_VALENCIA_01",
                  "TIENDA_VALENCIA_01", "TIENDA_BARCELONA_01"],
    "monto": [50.00, 175.25, 90.00, 90.00, 110.00],
    "fecha": ["2026-10-10", "2026-10-10", "2026-10-10", "2026-10-10", "2026-10-10"]
}
pd.DataFrame(ventas_dia3).to_csv(f"{raw_path}/ventas_20261010.csv", index=False)

# ── CSV 4: Ventas – Día 4  ──
ventas_dia4 = {
    "venta_id": ["V020", "V021"],
    "cliente_id": ["C020", "C999"],
    "tienda_id": ["TIENDA_MADRID_01", "TIENDA_BILBAO_01"],
    "monto": [150.00, 275.25],
    "fecha": ["2026-10-11", "2026-10-11"]
}
pd.DataFrame(ventas_dia4).to_csv(f"{raw_path}/ventas_20261011.csv", index=False)

print("4 CSV generados correctamente:")
print(f"  - {raw_path}/productos_20261009.csv")
print(f"  - {raw_path}/ventas_20261009.csv")
print(f"  - {raw_path}/ventas_20261010.csv")
print(f"  - {raw_path}/ventas_20261011.csv")


### Creamos  Volumen llamado "checkpoints" en el esquema bronze

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dev_casopractico.bronze.checkpoints;



In [0]:
%python
# Simular llegada al volumen raw de un cambio de datos (un cliente nuevo y una actualización)
import pandas as pd
raw_path = "/Volumes/dev_casopractico/bronze/raw"

clientes_nuevos = {
    "cliente_id": ["C002", "C005"], 
    "nombre": ["Ana Gómez Modificado", "Carlos Slim"], # C002 cambia de nombre, C005 es nuevo
    "email": ["ana.nueva@email.com", "carlos@email.com"]
}
pd.DataFrame(clientes_nuevos).to_csv(f"{raw_path}/clientes_updates.csv", index=False)
